# IndexTTS-vLLM on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keboqi/index-tts-vllm/blob/master/index_tts_vllm_colab.ipynb)

Run the IndexTTS 2.0 speech studio with voice cloning, speaker presets, emotion controls, and streaming audio.
Select **Runtime → Change runtime type → G4 GPU** before running cells 1–6 in order.
The notebook requests G4 in its metadata and verifies the assigned RTX PRO 6000 GPU; availability depends on Colab.

Adapted from the supplied MiniMax H3 notebook: isolated Python 3.12, rerunnable setup, optional Google Drive persistence, and live server logs.
This app uses **FastAPI**. Cell 6 prints a Cloudflare public URL after the model is ready; keep that cell running while using the studio.
Anyone with the public URL can access the app and its files. Stop the cell to close the server and tunnel.

Code, environments, checkpoints, and caches stay on the Colab runtime disk. Optional Drive storage persists outputs and speaker presets.
Local runtime files disappear when Colab deletes the runtime.

In [ ]:
# @title 1. Verify G4 GPU and system resources
import os
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIRECTORY = ""  # @param {type:"string"}
REQUIRE_G4 = True  # @param {type:"boolean"}
NOTEBOOK_DIR = Path(NOTEBOOK_DIRECTORY or globals().get("NOTEBOOK_DIR", Path.cwd())).expanduser().resolve()
if not NOTEBOOK_DIR.is_dir():
    raise FileNotFoundError(NOTEBOOK_DIR)
if sys.platform != "linux":
    raise RuntimeError("This notebook requires a Linux Colab GPU runtime.")
try:
    gpu_rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()
except (OSError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Select G4 GPU in Colab runtime settings, reconnect, and rerun.") from exc
if len(gpu_rows) != 1:
    raise RuntimeError(f"Expected one GPU, found: {gpu_rows}")
if REQUIRE_G4 and "RTX PRO 6000" not in gpu_rows[0].upper():
    raise RuntimeError(f"Expected Colab G4 / RTX PRO 6000, found: {gpu_rows[0]}")
print(f"GPU: {gpu_rows[0]}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
for line in Path("/proc/meminfo").read_text().splitlines():
    if line.startswith("MemAvailable:"):
        print(f"Available system RAM: {int(line.split()[1]) / 2**20:.1f} GiB")
print(f"Free disk: {shutil.disk_usage(NOTEBOOK_DIR).free / 2**30:.1f} GiB")


### 2. Clone the repository

Clone the repository's default branch, or choose a branch/tag with REPO_REF.
Reruns reuse a complete checkout. Enable UPDATE_REPOSITORY to fast-forward a clean checkout to the selected remote revision.
Incomplete child checkouts are preserved under a backup name. Existing local changes are never reset.

In [ ]:
# @title 2. Clone or reuse IndexTTS-vLLM
import signal
import uuid

REPO_URL = "https://github.com/keboqi/index-tts-vllm.git"  # @param {type:"string"}
REPO_REF = ""  # @param {type:"string"}
UPDATE_REPOSITORY = False  # @param {type:"boolean"}
# @markdown Clone optional TTS backends only. Their existing launchers install dependencies and download weights on first use.
INSTALL_CONFUCIUS = True  # @param {type:"boolean"}
INSTALL_INDEXTTS25 = True  # @param {type:"boolean"}

def stop_process_group(proc):
    # Signal even if the parent exited: workers may still belong to its group.
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait(timeout=5)

def run_logged(command, *, cwd=None, env=None):
    """Stream subprocess logs into Colab and stop child processes on interruption."""
    proc = subprocess.Popen(
        [str(part) for part in command], cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
        start_new_session=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        result = proc.wait()
        if result:
            raise subprocess.CalledProcessError(result, command)
    finally:
        stop_process_group(proc)
        proc.stdout.close()

REQUIRED_FILES = (
    "quickstart.sh", "fastapi_webui_v2.py", "fastapi_webui_v2_impl.py",
    "requirements-core.txt", "requirements-modal.txt", "requirements-clearvoice.txt", "constraints-main.txt",
    "indextts_web/config.py", "indextts/infer_vllm_v2.py", "index_new.html",
)

def is_complete_repository(directory):
    if not (directory / ".git").exists():
        return False
    result = subprocess.run(
        ["git", "-C", str(directory), "ls-tree", "-r", "--name-only", "-z", "HEAD"],
        capture_output=True, text=True,
    )
    files = [name for name in result.stdout.split("\0") if name]
    return (
        result.returncode == 0 and set(REQUIRED_FILES).issubset(files)
        and all((directory / name).is_file() for name in files)
    )

WORKSPACE_DIR = NOTEBOOK_DIR if is_complete_repository(NOTEBOOK_DIR) else NOTEBOOK_DIR / "index-tts-vllm"
if not is_complete_repository(WORKSPACE_DIR):
    if WORKSPACE_DIR.exists() or WORKSPACE_DIR.is_symlink():
        backup = WORKSPACE_DIR.with_name(f"index-tts-vllm-incomplete-{uuid.uuid4().hex[:8]}")
        WORKSPACE_DIR.rename(backup)
        print(f"Preserved incomplete checkout: {backup}")
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF.strip():
        command += ["--branch", REPO_REF.strip()]
    run_logged(command + [REPO_URL, WORKSPACE_DIR])
elif UPDATE_REPOSITORY:
    changes = subprocess.check_output(
        ["git", "-C", str(WORKSPACE_DIR), "status", "--porcelain", "--untracked-files=no"], text=True,
    )
    if changes.strip():
        raise RuntimeError("Checkout has local changes. Keep UPDATE_REPOSITORY off or handle them before updating.")
    run_logged(["git", "fetch", "--depth", "1", REPO_URL, REPO_REF.strip() or "HEAD"], cwd=WORKSPACE_DIR)
    run_logged(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=WORKSPACE_DIR)
if not is_complete_repository(WORKSPACE_DIR):
    raise RuntimeError("Checkout is incomplete. Inspect the clone logs before continuing.")
print(f"Workspace: {WORKSPACE_DIR}")
print("Revision:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=WORKSPACE_DIR, text=True).strip())


def provision_optional_repository(name, url, relative, entrypoint, enabled):
    destination = WORKSPACE_DIR.parent / relative
    if not enabled:
        print(f"{name} checkout disabled; select another TTS backend in the UI.")
        return
    if not destination.exists():
        run_logged(["git", "clone", "--depth", "1", url, str(destination)])
    if not (destination / ".git").exists() or not (destination / entrypoint).is_file():
        raise RuntimeError(f"Incomplete {name} checkout: {destination}. Review it before retrying.")
    print(f"{name}: source ready. Setup and model downloads run only when selected.")

provision_optional_repository(
    "Confucius4-TTS", "https://github.com/keboqi/Confucius4-TTS.git", "Confucius4-TTS",
    "scripts/run_fastapi_uv.sh", INSTALL_CONFUCIUS,
)
provision_optional_repository(
    "IndexTTS 2.5", "https://github.com/keboqi/indextts-2.5-vllm-omni-experiment.git",
    "index-tts-2.5-vllm-omni-experiment", "experiments/indextts25_backend_compat/serve_api.sh", INSTALL_INDEXTTS25,
)


### 3. Install the isolated GPU environment

Use Python 3.12 and the repository's constraints: vLLM 0.10.2, Torch/torchaudio 2.8.0, torchvision 0.23.0, and CUDA 12.8.
This keeps vLLM's binary dependencies together instead of installing the latest Torch first.
[vLLM's installation guide](https://docs.vllm.ai/en/v0.10.2/getting_started/installation/gpu.html) requires CUDA 12.8 or newer for Blackwell.

Optional main-process integrations install from the same manifest used by the repo's Modal deployment.
Stable Audio source is pinned and uses that shared runtime stack. Its six older metadata pins are reported as known overrides; all other dependency-check errors stop setup.
Optional environment preparation is off by default. ClearVoice, MOSS and Qwen3-ASR + OmniVAD install in isolated environments automatically on first use. Check PREPARE_CLEARVOICE, PREPARE_MOSS or PREPARE_QWEN_ASR to install that environment before launch. MOSS uses the local Transformers service without Docker. Model downloads are separate options in cell 5; no inference warmup runs unless enabled in cell 6. Confucius and IndexTTS 2.5 use the sibling checkouts from cell 2. Their normal standalone launchers create isolated environments and download models on first use.
First installation can take several minutes. Reruns reuse the environment; no Colab kernel restart is needed.

In [ ]:
# @title 3. Install Python 3.12 and compatible dependencies
INSTALL_OPTIONAL_FEATURES = True  # @param {type:"boolean"}
# @markdown Prepare optional environments now; otherwise install automatically on first use. Models stay unloaded.
PREPARE_CLEARVOICE = False  # @param {type:"boolean"}
PREPARE_MOSS = False  # @param {type:"boolean"}
PREPARE_QWEN_ASR = False  # @param {type:"boolean"}
INSTALL_FLASH_ATTENTION = True  # @param {type:"boolean"}

VENV_DIR = NOTEBOOK_DIR / "venv_index_tts"
CACHE_DIR = NOTEBOOK_DIR / ".cache_index_tts"
PYTHON = VENV_DIR / "bin/python"
RUN_ENV = os.environ.copy()
# Avoid importing Colab's preinstalled packages or notebook-only plotting backend.
for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND", "PIP_CONSTRAINT", "UV_CONSTRAINT"):
    RUN_ENV.pop(key, None)
RUN_ENV.update({
    "PYTHONUNBUFFERED": "1", "PYTHONNOUSERSITE": "1",
    "UV_NO_CACHE": "1", "PIP_NO_CACHE_DIR": "1",
    "HF_HOME": str(CACHE_DIR / "huggingface"),
    "TORCH_HOME": str(CACHE_DIR / "torch"),
    "TRITON_CACHE_DIR": str(CACHE_DIR / "triton"),
    "VLLM_CACHE_ROOT": str(CACHE_DIR / "vllm"),
    "CUDA_CACHE_PATH": str(CACHE_DIR / "cuda"),
    "TORCHINDUCTOR_CACHE_DIR": str(CACHE_DIR / "torchinductor"),
    "TORCHINDUCTOR_COMPILE_THREADS": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:512",
    "ORT_DISABLE_TELEMETRY": "1",
    "QWEN3_TTS_USE_FLASH_ATTENTION": str(int(INSTALL_FLASH_ATTENTION)),
})
if os.geteuid() != 0:
    raise RuntimeError("System dependency setup expects the standard root Colab runtime.")
run_logged(["apt-get", "update"], env=RUN_ENV)
run_logged(["apt-get", "install", "-y", "--no-install-recommends",
            "git", "curl", "ffmpeg", "sox", "libsox-fmt-all", "libsndfile1", "libstdc++6",
            "libgl1", "libglib2.0-0", "build-essential", "cmake", "pkg-config", "xz-utils"], env=RUN_ENV)
run_logged([sys.executable, "-m", "pip", "install", "--quiet", "uv"], env=RUN_ENV)
UV = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    run_logged(UV + ["venv", "--python", "3.12", "--seed", str(VENV_DIR)], env=RUN_ENV)
version = subprocess.check_output([str(PYTHON), "-c", "import sys; print('%s.%s' % sys.version_info[:2])"], env=RUN_ENV, text=True).strip()
if version != "3.12":
    raise RuntimeError(f"{VENV_DIR} uses Python {version}; choose a fresh VENV_DIR for Python 3.12.")
RUN_ENV["VIRTUAL_ENV"] = str(VENV_DIR)
RUN_ENV["PATH"] = str(VENV_DIR / "bin") + os.pathsep + RUN_ENV["PATH"]

# Select CUDA for Torch packages only; general dependencies still come from PyPI.
# A global PyTorch --index-url hides pinned dependencies such as filelock.
constraints = WORKSPACE_DIR / "constraints-main.txt"
pip_install = UV + ["pip", "install", "--python", str(PYTHON),
                    "-c", str(constraints), "--torch-backend", "cu128"]
run_logged(pip_install + ["torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0"], env=RUN_ENV)
manifest = "requirements-modal.txt" if INSTALL_OPTIONAL_FEATURES else "requirements-core.txt"
run_logged(pip_install + ["-r", str(WORKSPACE_DIR / manifest), "flashinfer-python", "json-repair", "yt-dlp[default]"],
           cwd=WORKSPACE_DIR, env=RUN_ENV)
if INSTALL_OPTIONAL_FEATURES:
    # Pin the audited Stable Audio source without replacing the constrained Torch stack.
    run_logged([str(PYTHON), "-m", "pip", "install", "--force-reinstall", "--no-deps", "--ignore-requires-python",
                "git+https://github.com/Stability-AI/stable-audio-tools.git@3241adba4fc2a85cf5b29d9eb68d42f40a28e820"], env=RUN_ENV)
    # Qwen-TTS and audio-separator share ONNX import paths: GPU bindings go last.
    run_logged(pip_install + ["--reinstall", "--no-deps", "onnxruntime-gpu"], env=RUN_ENV)
if INSTALL_FLASH_ATTENTION:
    abi = subprocess.check_output(
        [str(PYTHON), "-c", "import torch; print(str(torch._C._GLIBCXX_USE_CXX11_ABI).upper())"],
        text=True, env=RUN_ENV,
    ).strip()
    wheel = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3.post1/"
        f"flash_attn-2.8.3.post1+cu12torch2.8cxx11abi{abi}-cp312-cp312-linux_x86_64.whl"
    )
    run_logged(pip_install + ["--no-deps", wheel], env=RUN_ENV)
def check_dependencies(python, env):
    """Keep pip check strict except for six audited Stable Audio metadata overrides."""
    result = subprocess.run(
        [str(python), "-m", "pip", "check"], env=env,
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(result.stdout.strip(), flush=True)
        return
    # Stable Audio is installed at the pinned source revision with --no-deps.
    # These exact versions come from requirements-modal.txt / constraints-main.txt.
    overrides = {
        "importlib-resources": ("5.12.0", "7.1.0"),
        "PyWavelets": ("1.4.1", "1.10.0"),
        "sentencepiece": ("0.1.99", "0.2.2"),
        "torch": ("2.7.1", "2.8.0+cu128"),
        "torchaudio": ("2.7.1", "2.8.0+cu128"),
        "vector-quantize-pytorch": ("1.14.41", "1.31.6"),
    }
    allowed = {
        (f"stable-audio-tools 0.0.20 has requirement {name}=={required}, "
         f"but you have {name} {installed}.").casefold()
        for name, (required, installed) in overrides.items()
    }
    lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    unexpected = [line for line in lines if line.casefold() not in allowed]
    if result.returncode != 1 or not lines or unexpected or result.stderr.strip():
        print(result.stdout, end="", flush=True)
        print(result.stderr, end="", flush=True)
        raise subprocess.CalledProcessError(result.returncode, result.args)
    for line in lines:
        print(f"[Known Stable Audio metadata override] {line}", flush=True)
    print("Dependency check passed with the six documented Stable Audio overrides.", flush=True)

check_dependencies(PYTHON, RUN_ENV)
run_logged([str(PYTHON), "-c", """
import importlib.metadata as metadata
import torch
assert metadata.version("vllm") == "0.10.2"
assert torch.__version__.split("+")[0] == "2.8.0"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available(), "CUDA is unavailable in the isolated environment"
x = torch.randn(64, 64, device="cuda")
print("CUDA smoke test:", (x @ x).mean().item())
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
from indextts.infer_vllm_v2 import IndexTTS2
from qwen_tts import Qwen3TTSModel
print("IndexTTS/vLLM and Qwen3-TTS imports OK")
"""], cwd=WORKSPACE_DIR, env=RUN_ENV)
if INSTALL_FLASH_ATTENTION:
    run_logged([str(PYTHON), "-c", "import flash_attn; print('FlashAttention imports OK')"],
               cwd=WORKSPACE_DIR, env=RUN_ENV)
if INSTALL_OPTIONAL_FEATURES:
    # Import checks do not download weights or run model warmup.
    run_logged([str(PYTHON), "-c", """
import onnxruntime
assert "CUDAExecutionProvider" in onnxruntime.get_available_providers(), "ONNX Runtime GPU provider missing"
from audio_separator.separator import Separator
from omnivad import OmniVAD
import whisperx_pipeline as whisperx
import parakeet_pipeline as parakeet
assert whisperx.whisperx is not None
assert parakeet.is_parakeet_available()
from stable_audio3_manager import StableAudio3Manager
StableAudio3Manager("checkpoints/stable-audio-3")._raise_for_unsupported_stable_audio_tools()
print("Optional ASR, separation, ONNX GPU and Stable Audio imports OK")
"""], cwd=WORKSPACE_DIR, env=RUN_ENV)
print(f"Python environment ready: {PYTHON}")

def optional_runtime_command(kind):
    return [str(PYTHON), "-m", "indextts_web.infrastructure.optional_runtime", kind,
            "--workspace", str(WORKSPACE_DIR), "--root", str(NOTEBOOK_DIR), "--cache", str(CACHE_DIR)]


def write_optional_wrappers():
    import shlex
    directory = WORKSPACE_DIR / ".colab"
    directory.mkdir(exist_ok=True)
    for kind in ("clearvoice", "qwen-asr"):
        path = directory / (kind + "-python")
        command = optional_runtime_command(kind) + ["--exec-python", "--"]
        path.write_text("#!/usr/bin/env bash\nset -euo pipefail\nexec " + shlex.join(command) + ' "$@"\n', encoding="utf-8")
        path.chmod(0o755)
    manager = directory / "moss-start.sh"
    manager.write_text("#!/usr/bin/env bash\nset -euo pipefail\nexec " + shlex.join(
        optional_runtime_command("moss") + ["--start-moss"]
    ) + " 2>&1 | tee " + shlex.quote(str(directory / "moss-setup.log")) + "\n", encoding="utf-8")
    manager.chmod(0o755)


def setup_clearvoice(enabled):
    RUN_ENV["CLEARVOICE_PYTHON"] = str(WORKSPACE_DIR / ".colab/clearvoice-python")
    if enabled:
        run_logged(optional_runtime_command("clearvoice") + ["--force"], cwd=WORKSPACE_DIR, env=RUN_ENV)
    else:
        print("ClearVoice will install in its own environment on first use.")


def setup_asr(kind, enabled):
    if kind == "moss":
        RUN_ENV.pop("COLAB_MOSS_PYTHON", None)
        RUN_ENV.update({"MOSS_TRANSCRIBE_BACKEND": "http", "MOSS_TRANSCRIBE_MANAGE_BACKEND": "1",
                        "MOSS_TRANSCRIBE_MANAGER_SCRIPT": str(WORKSPACE_DIR / ".colab/moss-start.sh"),
                        "MOSS_TRANSCRIBE_SGLANG_URL": "http://127.0.0.1:8003",
                        "MOSS_TRANSCRIBE_DEVICE": "cuda:0",
                        "MOSS_TRANSCRIBE_MODEL": "OpenMOSS-Team/MOSS-Transcribe-Diarize"})
    else:
        RUN_ENV.update({"QWEN_OMNIVAD_PYTHON": str(WORKSPACE_DIR / ".colab/qwen-asr-python"),
                        "QWEN_OMNIVAD_MODEL_DIR": str(WORKSPACE_DIR / "checkpoints/qwen_omnivad"),
                        "QWEN_OMNIVAD_CACHE_DIR": str(CACHE_DIR / "qwen_omnivad")})
    if enabled:
        run_logged(optional_runtime_command(kind) + ["--force"], cwd=WORKSPACE_DIR, env=RUN_ENV)
        if kind == "moss":
            RUN_ENV["COLAB_MOSS_PYTHON"] = str(NOTEBOOK_DIR / "venv_index_tts_moss/bin/python")
            RUN_ENV["MOSS_TRANSCRIBE_MANAGE_BACKEND"] = "0"
    else:
        print(f"{kind} will install in its own environment on first use.")

write_optional_wrappers()
setup_clearvoice(PREPARE_CLEARVOICE)
setup_asr("moss", PREPARE_MOSS)
setup_asr("qwen-asr", PREPARE_QWEN_ASR)


def setup_node():
    # Match Modal's Node 22 runtime without modifying Colab's system Node installation.
    import hashlib
    import re
    import urllib.request
    node_root = CACHE_DIR / "node"
    candidates = [str(node_root / "bin/node"), shutil.which("node", path=RUN_ENV["PATH"])]
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            try:
                version = subprocess.check_output([candidate, "--version"], text=True, env=RUN_ENV).strip()
                if int(version.lstrip("v").split(".")[0]) >= 22:
                    RUN_ENV["YTDLP_NODE_PATH"] = candidate
                    return
            except (OSError, ValueError, subprocess.CalledProcessError):
                pass
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    base = "https://nodejs.org/dist/latest-v22.x/"
    with urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60) as response:
        checksums = response.read().decode("utf-8")
    checksum, filename = next(
        line.split() for line in checksums.splitlines()
        if re.fullmatch(r"[0-9a-f]{64}\s+node-v22\.[0-9]+\.[0-9]+-linux-x64\.tar\.xz", line)
    )
    archive = CACHE_DIR / filename
    run_logged(["curl", "--fail", "--location", "--retry", "3", base + filename,
                "--output", str(archive)], env=RUN_ENV)
    with archive.open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != checksum:
            raise RuntimeError("Node archive checksum mismatch; refusing to install.")
    node_root.mkdir(parents=True, exist_ok=True)
    run_logged(["tar", "-xJf", str(archive), "--strip-components=1", "-C", str(node_root)], env=RUN_ENV)
    node = str(node_root / "bin/node")
    run_logged([node, "--version"], env=RUN_ENV)
    RUN_ENV["YTDLP_NODE_PATH"] = node
    archive.unlink()

setup_node()


### 4. Optional Google Drive persistence

Enable to save outputs and speaker presets under MyDrive/IndexTTS-vLLM.
Checkpoints stay on runtime disk. Before linking a populated local folder, the notebook copies missing data to Drive and preserves the original folder as a local backup.
If both locations contain different copies of the same file, setup stops instead of overwriting either copy.
Run this cell before launching the app.

In [ ]:
# @title 4. (Optional) Save outputs and speaker presets to Google Drive
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER = "IndexTTS-vLLM"  # @param {type:"string"}

def merge_missing_files(source, destination):
    import filecmp
    for item in source.iterdir():
        target = destination / item.name
        if item.is_symlink():
            raise RuntimeError(f"Review local symlink before migration: {item}")
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            merge_missing_files(item, target)
        elif target.exists():
            if not target.is_file() or not filecmp.cmp(item, target, shallow=False):
                raise RuntimeError(f"Drive conflict: {target}. Resolve it before rerunning; local files are preserved.")
        else:
            shutil.copy2(item, target)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    mount_dir = NOTEBOOK_DIR / "drive"
    drive.mount(str(mount_dir))
    my_drive = (mount_dir / "MyDrive").resolve()
    DRIVE_ROOT = (my_drive / DRIVE_FOLDER).resolve()
    if DRIVE_ROOT == my_drive or not DRIVE_ROOT.is_relative_to(my_drive):
        raise ValueError("DRIVE_FOLDER must name a subfolder inside MyDrive.")
    for name in ("outputs", "speaker_presets"):
        local = WORKSPACE_DIR / name
        target = DRIVE_ROOT / name
        if local.is_symlink():
            if local.resolve() != target.resolve():
                raise RuntimeError(f"{local} already links elsewhere; restore or choose the original Drive folder.")
            print(f"Reusing {local} -> {target}")
            continue
        if local.exists() and not local.is_dir():
            raise RuntimeError(f"Expected a directory: {local}")
        target.mkdir(parents=True, exist_ok=True)
        if local.exists():
            merge_missing_files(local, target)
            backup = local.with_name(f"{name}-before-drive-{uuid.uuid4().hex[:8]}")
            local.rename(backup)
            print(f"Preserved local backup: {backup}")
        local.symlink_to(target, target_is_directory=True)
        print(f"{local} -> {target}")
else:
    for name in ("outputs", "speaker_presets"):
        local = WORKSPACE_DIR / name
        if local.is_symlink():
            raise RuntimeError(f"{local} is linked to Drive. Re-enable Drive mounting before continuing.")
        local.mkdir(parents=True, exist_ok=True)
    print("Using local runtime storage.")


### 5. Download model weights

Downloads are resumable through Hugging Face's local-directory metadata.
IndexTTS 2.0 and HY-MT download by default, matching quick start.
All extra model download options default off. Unchecked Voice Design, MOSS, Qwen ASR and ClearVoice models download on first use.
Checking a model option prepares its files ahead of launch. Stable Audio requires an authorized token and a selected gated model download.
For authenticated downloads, add a Colab secret named HF_TOKEN, allow notebook access, and enable USE_HF_SECRET.
Tokens are passed through the child environment and never written to notebook source or launcher files.
Some auxiliary weights may still download on first inference.

In [ ]:
# @title 5. Download models (no inference or warmup)
DOWNLOAD_HY_MT = True  # @param {type:"boolean"}
# @markdown Voice Design otherwise downloads from Hugging Face on first use.
DOWNLOAD_VOICE_DESIGN = False  # @param {type:"boolean"}
DOWNLOAD_MOSS_MODEL = False  # @param {type:"boolean"}
# @markdown Prefetch Qwen ASR/aligner/Sortformer and default ClearVoice models; these also prepare their isolated environments.
DOWNLOAD_QWEN_MODELS = False  # @param {type:"boolean"}
DOWNLOAD_CLEARVOICE_MODELS = False  # @param {type:"boolean"}
# @markdown Stable Audio needs access to the gated model and an HF_TOKEN secret. Off by default.
STABLE_AUDIO_DOWNLOAD = "off"  # @param ["off", "small-music", "small-sfx", "medium", "all"]
USE_HF_SECRET = False  # @param {type:"boolean"}

import json

if USE_HF_SECRET:
    from google.colab import userdata
    RUN_ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")
if STABLE_AUDIO_DOWNLOAD != "off" and not INSTALL_OPTIONAL_FEATURES:
    raise ValueError("Enable INSTALL_OPTIONAL_FEATURES in cell 3 before downloading Stable Audio models.")

MODEL_DIR = WORKSPACE_DIR / "checkpoints"
targets = ["index"]
if DOWNLOAD_HY_MT:
    targets.append("hy-mt")
if DOWNLOAD_VOICE_DESIGN:
    targets.append("voice-design")
if DOWNLOAD_MOSS_MODEL:
    targets.append("moss")
if STABLE_AUDIO_DOWNLOAD == "all":
    targets += ["stable-audio-" + variant for variant in ("medium", "small-music", "small-sfx")]
elif STABLE_AUDIO_DOWNLOAD != "off":
    targets.append("stable-audio-" + STABLE_AUDIO_DOWNLOAD)

# Use the same model catalog and shard validation as Modal's setup manager.
MODEL_DOWNLOAD_SOURCE = r"""
import json
import sys
from pathlib import Path
from huggingface_hub import snapshot_download
from indextts_web.infrastructure.model_setup import MODELS, missing_model_files
root = Path(sys.argv[1])
for target in json.loads(sys.argv[2]):
    model = MODELS[target]
    for repo, relative, patterns in model["downloads"]:
        print(f"Downloading/resuming {repo}", flush=True)
        snapshot_download(repo_id=repo, local_dir=str(root / relative), allow_patterns=patterns)
    missing = missing_model_files(root, model)
    if missing:
        raise RuntimeError(f"{model['name']} bundle incomplete: " + ", ".join(missing))
    print(f"{model['name']}: checkpoint files verified", flush=True)
"""
run_logged([str(PYTHON), "-u", "-c", MODEL_DOWNLOAD_SOURCE, str(WORKSPACE_DIR), json.dumps(targets)],
           cwd=WORKSPACE_DIR, env=RUN_ENV)
# Empty HY-MT override enables the shared pipeline's normal Hugging Face fallback.
RUN_ENV["HY_MT_TRANSLATION_LOCAL_DIR"] = str(MODEL_DIR / "hy-mt") if DOWNLOAD_HY_MT else ""
RUN_ENV["QWEN3_VOICE_DESIGN_MODEL"] = (
    str(MODEL_DIR / "Qwen3-TTS-12Hz-1.7B-VoiceDesign") if DOWNLOAD_VOICE_DESIGN
    else "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
)
RUN_ENV["MOSS_TRANSCRIBE_MODEL"] = (
    str(MODEL_DIR / "MOSS-Transcribe-Diarize") if DOWNLOAD_MOSS_MODEL
    else "OpenMOSS-Team/MOSS-Transcribe-Diarize"
)

if DOWNLOAD_QWEN_MODELS:
    run_logged([RUN_ENV["QWEN_OMNIVAD_PYTHON"], "-c", """
import qwen_omnivad_pipeline as pipeline
from huggingface_hub import snapshot_download
pipeline._resolve_hf_model_path(pipeline.QWEN_ASR_MODEL, label="Qwen ASR")
pipeline._resolve_hf_model_path(pipeline.QWEN_ASR_FORCED_ALIGNER, label="Qwen forced aligner")
assert pipeline._resolve_omnivad_model_path(), "OmniVAD model could not be staged"
snapshot_download(pipeline.QWEN_OMNIVAD_SORTFORMER_MODEL)
print("Qwen ASR, aligner, OmniVAD and Sortformer weights downloaded; no inference warmup.")
"""], cwd=WORKSPACE_DIR, env=RUN_ENV)
if DOWNLOAD_CLEARVOICE_MODELS:
    run_logged([RUN_ENV["CLEARVOICE_PYTHON"], "-c", """
from clearvoice import ClearVoice
for task, name in (("speech_enhancement", "MossFormerGAN_SE_16K"),
                   ("speech_super_resolution", "MossFormer2_SR_48K")):
    model = ClearVoice(task=task, model_names=[name])
    del model
print("Default ClearVoice models prepared; no inference warmup.")
"""], cwd=WORKSPACE_DIR, env=RUN_ENV)


### 6. Launch the speech studio

Cloudflare is enabled by default. The launch cell waits for /health to report ready, prepares any checked optional TTS services, then prints a trycloudflare.com URL.
PREPARE_CONFUCIUS_SERVICE and PREPARE_INDEXTTS25_SERVICE default off. Checking either runs its existing setup and model loading before publishing the URL, then sleeps its engines; it does not send synthesis requests.
[Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) are temporary; restarting produces a new URL.
If you disable the public tunnel, use Colab's authenticated port preview.

The app chooses its memory budget from detected VRAM.
Warmup is optional and **off by default**. Enable ENABLE_WARMUP to speed up generation,
but it costs time to warm up before the app becomes ready. This single option controls S2Mel compilation and startup warmup together: both off or both on.
Interrupt this cell to stop the app, workers, and tunnel before rerunning setup.

For Colab Terminal use, this cell also writes **run_index_tts_colab.sh** in the workspace.
Stop the notebook launch first, then run that script from the printed workspace path.
The terminal launcher reuses the environment and files; it does not contain the HF token.

In [ ]:
# @title 6. Launch IndexTTS-vLLM with a public URL
ENABLE_CLOUDFLARE_TUNNEL = True  # @param {type:"boolean"}
# @markdown Optionally prepare these services and their models before publishing the URL. Otherwise their normal launchers run on first use.
PREPARE_CONFUCIUS_SERVICE = False  # @param {type:"boolean"}
PREPARE_INDEXTTS25_SERVICE = False  # @param {type:"boolean"}
# @markdown One switch for warmup and S2Mel compilation: both off by default. Turning it on speeds up generation but costs startup warmup time.
ENABLE_WARMUP = False  # @param {type:"boolean"}
SERVER_PORT = 8000  # @param {type:"integer"}
STARTUP_TIMEOUT_SECONDS = 1800  # @param {type:"integer"}

import shlex

if SERVER_PORT == 8003:
    raise ValueError("Port 8003 is reserved for the on-demand MOSS service.")
if not 1 <= SERVER_PORT <= 65535:
    raise ValueError("SERVER_PORT must be between 1 and 65535.")
if STARTUP_TIMEOUT_SECONDS < 1:
    raise ValueError("STARTUP_TIMEOUT_SECONDS must be positive.")
for name in ("outputs", "speaker_presets"):
    local = WORKSPACE_DIR / name
    if not local.is_dir():
        raise RuntimeError(f"{local} is unavailable. Rerun cell 4, including Drive mounting if enabled.")

# Keep the app, automatic GPU profile, and child processes on the same setting.
RUN_ENV["INDEXTTS_USE_TORCH_COMPILE"] = str(int(ENABLE_WARMUP))
RUN_ENV["CONFUCIUS_USE_TORCH_COMPILE"] = str(int(ENABLE_WARMUP))
RUN_ENV["WARMUP"] = str(int(ENABLE_WARMUP))
RUN_ENV["COLAB_PREPARE_TTS"] = json.dumps([
    key for enabled, key in ((PREPARE_CONFUCIUS_SERVICE, "confucius_vllm"),
                             (PREPARE_INDEXTTS25_SERVICE, "indextts25_omni")) if enabled
])
COLAB_DIR = WORKSPACE_DIR / ".colab"
COLAB_DIR.mkdir(exist_ok=True)
CLOUDFLARED = COLAB_DIR / "cloudflared"
if ENABLE_CLOUDFLARE_TUNNEL and not CLOUDFLARED.is_file():
    partial = CLOUDFLARED.with_suffix(".partial")
    run_logged(["curl", "--fail", "--location", "--retry", "3",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "--output", str(partial)], env=RUN_ENV)
    partial.chmod(0o755)
    partial.replace(CLOUDFLARED)

LAUNCHER_SOURCE = r'''
import json
import os
from pathlib import Path
import queue
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

workspace = Path(__file__).resolve().parent.parent
port = int(sys.argv[1])
public = sys.argv[2] == "1"
warmup_enabled = sys.argv[3] == "1"
startup_timeout = int(sys.argv[4])
events = queue.Queue()
processes = []

def stop(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait(timeout=5)

def relay(proc, label):
    for line in proc.stdout:
        events.put((label, line))
    proc.stdout.close()

def start(command, label):
    proc = subprocess.Popen(
        command, cwd=workspace, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True,
    )
    processes.append(proc)
    threading.Thread(target=relay, args=(proc, label), daemon=True).start()
    return proc

def relay_managed_moss(filename):
    path = workspace / ".colab" / filename
    stream = None
    while True:
        if stream is None:
            try:
                stream = path.open(encoding="utf-8", errors="replace")
            except FileNotFoundError:
                time.sleep(0.5)
                continue
        line = stream.readline()
        if line:
            events.put(("moss", line))
        else:
            # The on-demand manager can truncate its log when a failed setup retries.
            try:
                if path.stat().st_size < stream.tell():
                    stream.seek(0)
            except FileNotFoundError:
                stream.close()
                stream = None
            time.sleep(0.5)


def prepare_tts_services():
    try:
        for key in json.loads(os.environ.get("COLAB_PREPARE_TTS", "[]")):
            events.put(("setup", f"Preparing {key} with its existing standalone launcher...\n"))
            # Wake starts the existing manager; unload sleeps its engines afterward.
            # No synthesis request or inference warmup is sent here.
            for action in ("wake", "unload"):
                request = urllib.request.Request(
                    f"http://127.0.0.1:{port}/api/models/{action}",
                    data=json.dumps({"model_key": key}).encode(),
                    headers={"Content-Type": "application/json"}, method="POST",
                )
                with urllib.request.urlopen(request, timeout=max(startup_timeout, 3600)) as response:
                    if json.load(response).get("status") != "success":
                        raise RuntimeError(f"{key}: {action} did not succeed")
            events.put(("setup", f"{key} prepared; engines sleeping until use.\n"))
    except Exception as exc:
        preparation_errors.append(exc)
    finally:
        preparation_done.set()


def server_command():
    # Override any inherited compilation setting with the notebook's one switch.
    os.environ["INDEXTTS_USE_TORCH_COMPILE"] = str(int(warmup_enabled))
    return [
        sys.executable, "-u", str(workspace / "fastapi_webui_v2.py"),
        "--host", "0.0.0.0", "--port", str(port),
        "--model_dir", str(workspace / "checkpoints"), "--tts_backend", "index",
        "--use_torch_compile" if warmup_enabled else "--no-use_torch_compile",
        "--confucius_repo_dir", str(workspace.parent / "Confucius4-TTS"),
        "--indextts25_repo_dir", str(workspace.parent / "index-tts-2.5-vllm-omni-experiment"),
        "--confucius_attach_stdio", "--indextts25_attach_stdio",
        "--no-confucius_detach_process", "--no-indextts25_detach_process",
    ]

def start_moss_service():
    python = os.environ.get("COLAB_MOSS_PYTHON")
    if not python:
        return None
    moss_port = 8003
    if port == moss_port:
        raise ValueError("SERVER_PORT 8003 is reserved for the local MOSS service. Choose another WebUI port.")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", moss_port))
    # The same process list owns MOSS, the app and the tunnel, including failed startup.
    proc = start([python, "-m", "uvicorn", "moss_transcribe_server:app",
                  "--host", "127.0.0.1", "--port", str(moss_port)], "moss")
    deadline = time.monotonic() + min(startup_timeout, 120)
    while time.monotonic() < deadline:
        try:
            label, line = events.get(timeout=0.2)
            print(f"[{label}] {line}", end="", flush=True)
        except queue.Empty:
            pass
        if proc.poll() is not None:
            raise RuntimeError(f"MOSS service exited with code {proc.returncode}; inspect the moss logs above.")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8003/model/status", timeout=1) as response:
                status = json.load(response)
                if status.get("service") == "moss-transcribe":
                    print("MOSS service ready; weights load on first transcription or model wake.", flush=True)
                    return proc
        except (OSError, ValueError, urllib.error.HTTPError):
            pass
    raise TimeoutError("MOSS service startup timed out; inspect the moss logs above.")


def interrupted(signum, frame):
    raise KeyboardInterrupt

# Catch outer notebook cancellation so separately grouped app workers are cleaned up.
signal.signal(signal.SIGTERM, interrupted)
signal.signal(signal.SIGINT, interrupted)
try:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", port))
    for filename in ("moss-managed.log", "moss-setup.log"):
        threading.Thread(target=relay_managed_moss, args=(filename,), daemon=True).start()
    moss_server = start_moss_service()
    command = server_command()
    state = "ON" if warmup_enabled else "OFF"
    print(f"Warmup and S2Mel compilation: {state}", flush=True)
    server = start(command, "app")
    tunnel = None
    ready = False
    announced = False
    preparation_done = threading.Event()
    preparation_errors = []
    started = time.monotonic()
    next_check = 0
    print("Starting IndexTTS. Keep this cell running; stop it to close the app.", flush=True)
    while True:
        try:
            label, line = events.get(timeout=0.2)
            print(f"[{label}] {line}", end="", flush=True)
        except queue.Empty:
            pass
        if moss_server is not None and moss_server.poll() is not None:
            raise RuntimeError(f"MOSS service exited with code {moss_server.returncode}; inspect the moss logs above.")
        if server.poll() is not None:
            raise RuntimeError(f"WebUI exited with code {server.returncode}; inspect the app logs above.")
        if tunnel is not None and tunnel.poll() is not None:
            raise RuntimeError(f"Cloudflare exited with code {tunnel.returncode}; inspect the tunnel logs above.")
        now = time.monotonic()
        if not ready and now >= next_check:
            next_check = now + 3
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=1) as response:
                    ready = bool(json.load(response).get("ready"))
            except (OSError, ValueError, urllib.error.HTTPError):
                pass
            if ready:
                threading.Thread(target=prepare_tts_services, daemon=True).start()
            elif now - started > startup_timeout:
                raise TimeoutError("Model startup timed out. Inspect the app logs or increase the timeout.")
        if preparation_errors:
            raise RuntimeError(f"Requested TTS service preparation failed: {preparation_errors[0]}")
        if ready and preparation_done.is_set() and not announced:
            announced = True
            print(f"\nWebUI ready at http://127.0.0.1:{port}", flush=True)
            if public:
                tunnel = start([
                    str(workspace / ".colab/cloudflared"), "tunnel",
                    "--url", f"http://127.0.0.1:{port}", "--no-autoupdate", "--protocol", "http2",
                ], "tunnel")
                print("Open the https://...trycloudflare.com URL in the tunnel logs below.", flush=True)
except KeyboardInterrupt:
    print("\nStopping MOSS, IndexTTS and the public tunnel...", flush=True)
finally:
    # Do not let a second notebook stop interrupt cleanup.
    signal.signal(signal.SIGTERM, signal.SIG_IGN)
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    for proc in reversed(processes):
        stop(proc)
'''
(COLAB_DIR / "serve.py").write_text(LAUNCHER_SOURCE, encoding="utf-8")
launcher = WORKSPACE_DIR / "run_index_tts_colab.sh"
persistent_env = {
    key: RUN_ENV[key] for key in (
        "HF_HOME", "TORCH_HOME", "TRITON_CACHE_DIR", "VLLM_CACHE_ROOT",
        "HY_MT_TRANSLATION_LOCAL_DIR", "PYTHONUNBUFFERED", "PYTHONNOUSERSITE",
        "CLEARVOICE_PYTHON", "INDEXTTS_USE_TORCH_COMPILE",
        "YTDLP_NODE_PATH", "QWEN3_VOICE_DESIGN_MODEL", "QWEN3_TTS_USE_FLASH_ATTENTION",
        "CUDA_CACHE_PATH", "TORCHINDUCTOR_CACHE_DIR", "TORCHINDUCTOR_COMPILE_THREADS",
        "PYTORCH_CUDA_ALLOC_CONF", "ORT_DISABLE_TELEMETRY",
        "CONFUCIUS_USE_TORCH_COMPILE", "WARMUP", "COLAB_PREPARE_TTS",
        "COLAB_MOSS_PYTHON", "MOSS_TRANSCRIBE_BACKEND", "MOSS_TRANSCRIBE_MANAGE_BACKEND",
        "MOSS_TRANSCRIBE_SGLANG_URL", "MOSS_TRANSCRIBE_DEVICE", "MOSS_TRANSCRIBE_MODEL",
        "MOSS_TRANSCRIBE_MANAGER_SCRIPT",
        "QWEN_OMNIVAD_PYTHON", "QWEN_OMNIVAD_MODEL_DIR", "QWEN_OMNIVAD_CACHE_DIR",
    ) if key in RUN_ENV
}
launcher_text = "#!/usr/bin/env bash\nset -euo pipefail\n"
launcher_text += "unset PYTHONPATH PYTHONHOME MPLBACKEND CLEARVOICE_PYTHON COLAB_MOSS_PYTHON QWEN_OMNIVAD_PYTHON\n"
launcher_text += f"export VIRTUAL_ENV={shlex.quote(str(VENV_DIR))}\n"
launcher_text += f'export PATH={shlex.quote(str(VENV_DIR / "bin"))}:"$PATH"\n'
for key, value in persistent_env.items():
    launcher_text += f"export {key}={shlex.quote(value)}\n"
launcher_text += "exec " + shlex.join([
    str(PYTHON), "-u", str(COLAB_DIR / "serve.py"), str(SERVER_PORT),
    str(int(ENABLE_CLOUDFLARE_TUNNEL)), str(int(ENABLE_WARMUP)),
    str(STARTUP_TIMEOUT_SECONDS),
]) + "\n"
launcher.write_text(launcher_text, encoding="utf-8")
launcher.chmod(0o755)
print(f"Terminal launcher: bash {shlex.quote(str(launcher))}", flush=True)
if not ENABLE_CLOUDFLARE_TUNNEL:
    from google.colab import output
    output.serve_kernel_port_as_window(SERVER_PORT)
run_logged(["bash", str(launcher)], cwd=WORKSPACE_DIR, env=RUN_ENV)


### Using the studio

- Open the public URL, upload a short reference voice or save a speaker preset, then enter text and synthesize with **IndexTTS 2.0**.
- Download generated files from the UI. When Drive is enabled, outputs and presets persist under MyDrive/IndexTTS-vLLM.
- Optional main-process integrations include WhisperX/Parakeet transcription, audio separation, video downloads, Qwen3 voice design, and Stable Audio runtime dependencies. ASR and separation weights download on first use. Voice Design uses the Hugging Face model by default, or can be downloaded in cell 5. Stable Audio needs gated access and the optional download in cell 5.
- ClearVoice reference enhancement and super-resolution use an isolated worker. By default, its environment and weights install on first use; PREPARE_CLEARVOICE installs the environment before launch.
- MOSS and Qwen3-ASR install automatically on first use. Their PREPARE options in cell 3 default off; enabling them prepares the isolated environments before launch. The local MOSS service starts on demand unless PREPARE_MOSS is enabled. First use can take several minutes.
- Confucius and IndexTTS 2.5 sources are cloned by default in cell 2. Selecting either backend triggers its normal standalone setup and model downloads; this can take several minutes. Their progress appears in the app logs. To prepare one before publishing the URL, enable its PREPARE service option in cell 6.
- Gated model access, cloud translation keys and site-specific download credentials must be supplied by the user. See [Colab setup and deployment parity](https://github.com/keboqi/index-tts-vllm/blob/master/docs/colab_setup_parity.md) for details.
- After reconnecting to a new runtime, rerun cells 1–6. To restart within the same runtime, stop cell 6 and rerun it. To change Drive settings, stop the app first.
- If setup fails, fix the first reported error and rerun that cell. The notebook does not modify quickstart.sh or upgrade vLLM beyond the repository pin.